# Rush Hour MCTS + LLM Optimization Pipeline

Configuration:
1. **`mcts/hyperparams/rush_hour_hyperparams.py`** — engine params, game identity,
   optimization settings
2. **`mcts/training_logic/rush_hour_training.py`** — puzzles, mastery criteria,
   puzzle-selection strategy

The orchestrator (`OptimizationRunner.from_config("rush_hour_hyperparams.py")`)
reads both files and drives the iterative LLM optimization loop.


## Cell 1: Setup — imports & quick sanity check

In [6]:
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mcts import MCTSEngine
from mcts.games import RushHour
from mcts.games.rush_hour import PUZZLES

print(f"Working dir : {Path('.').resolve()}")
print(f"Root        : {ROOT}")
print(f"Puzzles     : {list(PUZZLES.keys())}")
print("All imports OK ✓")

Working dir : /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/scripts
Root        : /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation
Puzzles     : ['easy1', 'easy2', 'easy3', 'medium1', 'medium2', 'hard1', 'hard2', 'hard3']
All imports OK ✓


## Cell 2: Display a puzzle

Show the initial board state and basic metrics for a chosen puzzle.

In [7]:
PUZZLE = "easy2"  # <<< Change puzzle here

game = RushHour(puzzle_name=PUZZLE, max_moves=300)
state = game.new_initial_state()
print(state)
print(f"\nLegal actions ({len(state.legal_actions())}): {state.legal_actions()}")

Move 0/300 | Primary dist: 4 | Blocking: 1 | Pieces: 5
..BBCC
..D...
AAD...
..D...
..EE..
......

Legal actions (6): [(1, -1), (1, -2), (4, -1), (4, -2), (4, 1), (4, 2)]


## Cell 3: Baseline — play one game with default MCTS

In [ ]:
# Reasonable defaults for Rush Hour baseline
ITERS = 200
MAX_DEPTH = 500
MAX_MOVES = 80   # tight budget — enough for easy/medium, caps slow hard puzzles
LEVEL = "easy2"  # start puzzle

game = RushHour(puzzle_name=LEVEL, max_moves=MAX_MOVES)
engine = MCTSEngine(game, iterations=ITERS, max_rollout_depth=MAX_DEPTH, logging=True)
baseline = engine.play_game()

tag = "SOLVED" if baseline.get("solved") else "UNSOLVED"
print(f"Baseline ({LEVEL}): {tag} in {baseline.get('steps', '?')} moves  "
      f"returns={baseline.get('returns', '?')}")
print(f"Hyperparams: iterations={ITERS}, max_depth={MAX_DEPTH}")
print(f"Trace: {baseline.get('log_file', 'N/A')}")

Baseline (easy2): SOLVED in 239 moves  returns=[1.0]
Hyperparams: iterations=200, max_depth=500
Trace: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/mcts/records/RushHour (easy2)_20260307_122812_204484.json


## Cell 4: Batch evaluation — baseline across all puzzles

In [ ]:
import time

NUM_GAMES = 3
EVAL_ITERS = 100

# Per-tier move budgets — avoids wasting minutes on unsolvable puzzles
_TIER_MAX_MOVES = {"easy": 30, "medium": 80, "hard": 80}

def _tier(name):
    return name.rstrip("0123456789")

print(f"{'Puzzle':<10} {'MaxMv':>6} {'Solve%':>8} {'AvgSteps':>10} {'Time':>8}")
print("─" * 46)

for pname in PUZZLES:
    mm = _TIER_MAX_MOVES[_tier(pname)]
    g = RushHour(puzzle_name=pname, max_moves=mm)
    eng = MCTSEngine(g, iterations=EVAL_ITERS, max_rollout_depth=200)
    t0 = time.time()
    stats = eng.play_many(num_games=NUM_GAMES)
    elapsed = time.time() - t0
    print(f"{pname:<10} {mm:>6} {stats['solve_rate']*100:>7.0f}% "
          f"{stats['avg_steps']:>10.1f} {elapsed:>7.1f}s")

Puzzle       Solve%   AvgSteps     Time
────────────────────────────────────────
  Game 1/3: solved in 258 steps
  Game 2/3: solved in 261 steps
  Game 3/3: solved in 267 steps
easy1          100%      262.0     7.2s
  Game 1/3: solved in 253 steps
  Game 2/3: solved in 240 steps
  Game 3/3: solved in 248 steps
easy2          100%      247.0    12.8s
  Game 1/3: solved in 228 steps
  Game 2/3: solved in 234 steps
  Game 3/3: solved in 233 steps
easy3          100%      231.7    16.4s


KeyboardInterrupt: 

## Cell 5: Run the LLM optimization pipeline

This cell patches `default_hyperparams.py` to point at Rush Hour,
then runs the orchestrator. Make sure Ollama is running (`ollama serve`).

In [ ]:
# default_hyperparams.py should already be configured for Rush Hour.
# If it still points at Sokoban, patch it temporarily.
hp_path = ROOT / "MCTS_tools" / "hyperparams" / "default_hyperparams.py"
hp_src = hp_path.read_text()
_original_hp_src = hp_src

needs_patch = 'GAME_NAME = "sokoban"' in hp_src
if needs_patch:
    patches = {
        'GAME_NAME = "sokoban"':           'GAME_NAME = "rush_hour"',
        'GAME_CLASS = "Sokoban"':          'GAME_CLASS = "RushHour"',
        'TRAINING_LOGIC = "sokoban_training"':     'TRAINING_LOGIC = "rush_hour_training"',
    }
    for old, new in patches.items():
        hp_src = hp_src.replace(old, new)
    # Also ensure max_moves is set (not max_steps)
    if '"max_steps"' in hp_src:
        import re
        hp_src = re.sub(r'CONSTRUCTOR_KWARGS\s*=\s*\{[^}]+\}',
                        'CONSTRUCTOR_KWARGS = {"max_moves": 80}', hp_src)
    hp_path.write_text(hp_src)
    print("Patched default_hyperparams.py → Rush Hour")
else:
    print("default_hyperparams.py already configured for Rush Hour")

# ── Run the optimization ────────────────────────────────────────────
from orchestrator import OptimizationRunner

runner = OptimizationRunner.from_config(verbose=True)
summary = runner.run()

best_fns = summary["best_fns"]
print(f"\nbest_fns: { {p: ('set' if f else 'None') for p, f in best_fns.items()} }")
print(f"Final hyperparams: {summary.get('current_hyperparams', {})}")

# ── Restore original if we patched ──────────────────────────────────
if needs_patch:
    hp_path.write_text(_original_hp_src)
    print("Restored default_hyperparams.py → Sokoban")

Patched default_hyperparams.py → Rush Hour
Starting level: easy2
Hyperparams: iterations=200, max_depth=500, C=1.410
Phases to optimize: ['simulation', 'hyperparams', 'expansion']
  Computing baseline for easy2…
    easy2: composite=1.0000, solve_rate=100%, avg_returns=1.0000 (12.8s)
  ⏳ easy2 hit 100% on 3 runs — confirming with 7 more games…
  🎓 easy2 MASTERED — 10/10 solved (29.3s)
  Reject floor for easy2: 0.5000
  Active levels: ['easy1', 'easy2', 'easy3', 'medium1', 'medium2', 'hard1', 'hard2', 'hard3']

############################################################
  ITERATION 1/5, LEVEL=easy2, PHASE=expansion
  Baseline composite=1.0000, reject_floor=0.5000
  Active levels: 8/8, mastered: ['easy2']
############################################################
  Play: SOLVED in 257 steps  returns=1.0000  (4.4s)
Step 1/6: Building analysis prompt…
Step 2/6: Querying LLM (step 2 — draft code)…
Step 3/6: Querying LLM (step 3 — critique & finalize)…
  LLM query: status=success  elapsed

KeyboardInterrupt: 

## Cell 6: Comparative evaluation — baseline vs. optimized

In [ ]:
best_fns = summary["best_fns"]
ev = runner.evaluator
opt_tools = {p: f for p, f in best_fns.items() if f is not None} or None

has_optimized = opt_tools is not None
print(f"Best fns: { {p: ('set' if f else 'None') for p, f in best_fns.items()} }")
print(f"Final hyperparams: {summary.get('current_hyperparams', {})}")
print(f"Mastered: {sorted(summary.get('mastered_levels', set()))}")

if not has_optimized:
    print("\nNo optimized functions adopted — skipping comparative eval.")
else:
    eval_levels = sorted(ev.level_baselines.keys())
    print(f"\nEvaluating {len(eval_levels)} puzzles (n=3 each)…")

    rows = []
    for lvl in eval_levels:
        avg_b, sr_b, steps_b, _, t_b = ev.multi_eval(None, lvl, n=3, logging=False)
        avg_o, sr_o, steps_o, _, t_o = ev.multi_eval(
            None, lvl, n=3, logging=False, extra_tools=opt_tools,
        )
        rows.append((lvl, sr_b, sr_o, avg_b, avg_o, steps_b, steps_o, t_b, t_o))
        print(f"{lvl}: baseline={avg_b:.3f} ({sr_b:.0%})  "
              f"optimized={avg_o:.3f} ({sr_o:.0%})  [{t_b:.1f}s + {t_o:.1f}s]")

    print(f"\n{'Puzzle':<10} {'Base Solve%':>12} {'Opt Solve%':>12} "
          f"{'Base AvgRet':>12} {'Opt AvgRet':>12}")
    print("─" * 60)
    for lvl, sr_b, sr_o, avg_b, avg_o, *_ in rows:
        print(f"{lvl:<10} {sr_b*100:>11.0f}% {sr_o*100:>11.0f}% "
              f"{avg_b:>12.3f} {avg_o:>12.3f}")